# Advertised 20-hour battery life: test from a summary

**One-Sample Test for the Mean** · a worked example from the StatNex teaching guide

**The story.** A phone maker advertises "20 hours of battery life". A consumer magazine tested 30 phones and published only a summary of its test: 30 phones, mean 18.63 hours, standard deviation 1.58 hours.

**The question.** Can a reader check the magazine's conclusion from the published summary alone?

**To run it:**

1. Choose **Copy to Drive** at the top (or **File → Save a copy in Drive**). Your copy opens in a new tab. It is yours, so Colab runs it without a warning, and it keeps your changes.
2. In your copy, choose **Runtime → Run all**.

You can also run this page without a copy. Colab then warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `02_one_sample/one_sample_mean_test__summary.csv` (3 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Variable,Battery life (hours)
Sample size,30.0
Mean,18.63
Sample standard deviation,1.5807

These cells save `cohen_d_interval.py`: a step that no library does, written out. The analysis imports it.

In [ ]:
%%writefile cohen_d_interval.py
"""Cohen's d with its noncentral-t interval, and Hedges' exact small-sample correction.

SciPy has the noncentral t distribution (stats.nct) but no function that inverts it for
an interval, so the inversion is written out (Steiger & Fouladi 1997). The interval is
for δ = t * scale: 1/sqrt(n) for one sample or paired differences, sqrt(1/n1 + 1/n2) for
two groups with the pooled standard deviation.
"""

import math

import numpy as np
from scipy import optimize, special, stats


def _limit(t, df, q):
    """The noncentrality under which the observed t sits at the q quantile of t(df, ncp).

    The cdf at t falls as the noncentrality rises, so the bracket opens outwards from t,
    doubling, until it holds the root.
    """
    def gap(ncp):
        return stats.nct.cdf(t, df, ncp) - q

    step = 1.0
    while gap(t - step) <= 0:
        step *= 2
    low = t - step
    step = 1.0
    while gap(t + step) >= 0:
        step *= 2
    return optimize.brentq(gap, low, t + step, xtol=1e-14, rtol=4 * np.finfo(float).eps,
                           maxiter=500)


def cohen_d_interval(d, df, scale, level):
    """(lower, upper): the level interval for d = t * scale."""
    t = d / scale
    return _limit(t, df, (1 + level) / 2) * scale, _limit(t, df, (1 - level) / 2) * scale


def hedges_j(df):
    """J = Γ(ν/2) / (sqrt(ν/2) Γ((ν - 1)/2)), Hedges (1981); g = J * d."""
    return math.exp(special.gammaln(df / 2) - special.gammaln((df - 1) / 2)) / math.sqrt(df / 2)

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:.6g}".format)


def stars(p):
    """* for P < 0.05, ** for P < 0.01."""
    return "**" if p < 0.01 else "*" if p < 0.05 else ""

## The data

In [ ]:
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
variables = ["Battery life (hours)"]
# The population mean under the null hypothesis.
mu0 = 20

## Test for the mean, one column at a time

In [ ]:
# The statistic counts how many standard errors the sample mean lies from mu0. If the
# true mean were mu0, the P value is how often a sample would land at least that far
# away by chance. A small P says the gap is unlikely to be luck; it does not say how
# big or important the gap is (the effect sizes below do that).
results = {}
for name in variables:
    # Summary data: each column holds the sample size, the mean and the standard
    # deviation, top to bottom.
    size, mean, sd = pd.to_numeric(data[name], errors="coerce").dropna().to_numpy(float)[:3]
    n = int(round(size))
    se = sd / np.sqrt(n)
    statistic = (mean - mu0) / se
    # t = (mean - mu0) / SE on n - 1 degrees of freedom. It assumes independent values
    # from a roughly normal population, or enough of them for the mean to be close to
    # normal (the central limit theorem).
    df = n - 1
    # A summary has no values to give stats.ttest_1samp, so the same t is read against
    # the t distribution directly.
    p_two = 2 * stats.t.sf(abs(statistic), df)
    # The one-sided P value is half the two-sided one, in the direction the data lie.
    # Use it only if that direction was chosen before seeing the data.
    p_one = p_two / 2
    results[name] = {"n": n, "variance": sd**2, "SD": sd, "SE": se, "mean": mean,
                     "mu0": mu0, "difference": mean - mu0, "statistic": statistic,
                     "df": df, "P two-sided": p_two, "P one-sided": p_one}

table = pd.DataFrame(results)
print(table)
print()
print(table.loc[["P two-sided", "P one-sided"]].map(stars))

## Effect sizes, with 95% confidence intervals

In [ ]:
# Cohen's d = (mean - mu0) / SD. Its interval inverts the noncentral t distribution at
# the observed t (Steiger & Fouladi 1997); Hedges' g = J * d removes d's small-sample
# bias, and its interval is d's times J. Both steps are in cohen_d_interval.py.
from cohen_d_interval import cohen_d_interval, hedges_j


def size(value):
    """Cohen's (1988) reading of |d|: 0.2 small, 0.5 medium, 0.8 large."""
    return ["negligible", "small", "medium", "large"][sum(abs(value) >= s for s in (0.2, 0.5, 0.8))]


rows = {}
for name in variables:
    n, d = results[name]["n"], results[name]["difference"] / results[name]["SD"]
    lower, upper = cohen_d_interval(d, n - 1, 1 / np.sqrt(n), 0.95)
    j = hedges_j(n - 1)
    rows[(name, "Cohen's d")] = [d, lower, upper, size(d)]
    rows[(name, "Hedges' g")] = [j * d, j * lower, j * upper, size(j * d)]
effect_sizes = pd.DataFrame.from_dict(rows, orient="index",
                                      columns=["value", "lower", "upper", "size"])
print()
print(effect_sizes)

## What the result says

This run uses only the three published numbers: 30 phones, a mean of 18.63 hours and a standard deviation of 1.58 hours. The standard deviation is the typical distance of one phone from the mean.

They are enough to redo the test. The mean is 1.37 hours below the advertised 20. The t statistic, the gap divided by its standard error, is -4.75, and the two-sided P value is 0.00005. The P value is the chance of a gap at least this large if the true average were 20 hours. Cohen's d, the gap measured in standard deviations, is -0.87, a large effect.

The verdict matches the magazine's: the phones fall well short of 20 hours. But the summary allows no normality test, so a reader cannot check whether the test's assumptions hold.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.